In [1]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import SolverFactory
from pyomo.environ import *


In [7]:
custo_demanda_produto = pd.read_excel('app21.ipynb.xlsx', sheet_name='custo_prod')
custo_demanda_produto.set_index('vinhedo', inplace=True)
custo_demanda_produto

,capacidade,custo
vinhedo,,
1,3500,23
2,3100,25


In [8]:
lucro_demanda = pd.read_excel('app21.ipynb.xlsx', sheet_name='l_d')
lucro_demanda.set_index('restaurante', inplace=True)
lucro_demanda

,demanda,preco
restaurante,,
1,1800,69
2,2300,67
3,1250,70
4,1750,66


In [9]:
distancia = pd.read_excel('app21.ipynb.xlsx', sheet_name='c_distancia')
distancia.set_index('vinhedo', inplace=True)
distancia

,1,2,3,4
vinhedo,,,,
1,7,8,13,9
2,12,6,8,7


In [33]:
model = pyo.ConcreteModel()

model.restaurantes = pyo.Set(initialize=lucro_demanda.index)
model.vinhedos = pyo.Set(initialize=custo_demanda_produto.index)

model.x = pyo.Var( model.vinhedos,model.restaurantes, bounds=(0, None), domain=pyo.NonNegativeReals)

def objetivo(model):
    return sum(model.x[v, r] * (lucro_demanda.loc[r, 'preco'] - custo_demanda_produto.loc[v, 'custo'] - distancia.loc[v, r]) for r in model.restaurantes for v in model.vinhedos)
model.objetivo = pyo.Objective(rule=objetivo, sense=pyo.maximize)

def restricao_demanda(model, r):
    return sum(model.x[v, r] for v in model.vinhedos) <= lucro_demanda.loc[r, 'demanda']
model.restricao_demanda = pyo.Constraint(model.restaurantes, rule=restricao_demanda)

def restricao_capacidade(model, v):
    return sum(model.x[v, r] for r in model.restaurantes) <= custo_demanda_produto.loc[v, 'capacidade']
model.restricao_capacidade = pyo.Constraint(model.vinhedos, rule=restricao_capacidade)

In [34]:
model.pprint()

2 Set Declarations
    restaurantes : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {1, 2, 3, 4}
    vinhedos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {1, 2}

1 Var Declarations
    x : Size=8, Index=vinhedos*restaurantes
        Key    : Lower : Value : Upper : Fixed : Stale : Domain
        (1, 1) :     0 :  None :  None : False :  True : NonNegativeReals
        (1, 2) :     0 :  None :  None : False :  True : NonNegativeReals
        (1, 3) :     0 :  None :  None : False :  True : NonNegativeReals
        (1, 4) :     0 :  None :  None : False :  True : NonNegativeReals
        (2, 1) :     0 :  None :  None : False :  True : NonNegativeReals
        (2, 2) :     0 :  None :  None : False :  True : NonNegativeReals
        (2, 3) :     0 :  None :  None : False :  True : NonNegativeReals
        (2, 4) :     0 :  None : 

In [37]:
opt = SolverFactory('cplex')
res = opt.solve(model,tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmpi819a9cm.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmpl3t8ip6v.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmpl3t8ip6v.pyomo.lp
Objective sense      : Maximize
Variables            :       8
Objective nonzeros   :       8
Linear constraints   :       6  [Less: 6]
  Nonzeros           :      16
  RHS nonzeros       :       6

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 32.00000         Max   : 39.00000    

In [41]:
for m,v in model.x:
    print(f'Vinhedo {m} para Restaurante {v}: {model.x[m,v].value}')

print(f'Lucro Total: R$ {model.objetivo():.2f}')


Vinhedo 1 para Restaurante 1: 1800.0
Vinhedo 1 para Restaurante 2: 450.0
Vinhedo 1 para Restaurante 3: 0.0
Vinhedo 1 para Restaurante 4: 1250.0
Vinhedo 2 para Restaurante 1: 0.0
Vinhedo 2 para Restaurante 2: 1850.0
Vinhedo 2 para Restaurante 3: 1250.0
Vinhedo 2 para Restaurante 4: 0.0
Lucro Total: R$ 241750.00
